In [3]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/CEAS_08.csv")

print(df.shape)
df.head()

(39154, 7)


,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1


### Accessing Files from Google Drive

Since the file `CEAS_08.csv` was not found, and you mentioned you'd download it from Drive, let's mount your Google Drive so Colab can access it. Then, you can provide the correct path to your CSV file.

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


After running the cell above and following the authentication steps, your Google Drive will be mounted at `/content/drive`.

You can then locate your `CEAS_08.csv` file within your Google Drive. For example, if it's directly in your 'MyDrive' folder, the path would be `/content/drive/MyDrive/CEAS_08.csv`. If it's in a subfolder like 'Colab Notebooks', the path might be `/content/drive/MyDrive/Colab Notebooks/CEAS_08.csv`.

Please update the `pd.read_csv` line in the original cell (`hcp6CVZ00bRs`) with the correct path to your file in Google Drive, then re-run that cell.

In [4]:
print(df.columns.tolist())
print(df.dtypes)


['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
sender      object
receiver    object
date        object
subject     object
body        object
label        int64
urls         int64
dtype: object


In [5]:
print(df['label'].value_counts())


label
1    21842
0    17312
Name: count, dtype: int64


In [47]:
df["combined_text"] = df["subject"].fillna("") + " " + df["body"].fillna("")
print(df["combined_text"].head())


0    Never agree to be a loser Buck up, your troubl...
1    Befriend Jenna Jameson \nUpgrade your sex and ...
2    CNN.com Daily Top 10 >+=+=+=+=+=+=+=+=+=+=+=+=...
3    Re: svn commit: r619753 - in /spamassassin/tru...
4    SpecialPricesPharmMoreinfo \nWelcomeFastShippi...
Name: combined_text, dtype: object


In [58]:
import pandas as pd

train_df = pd.read_csv("/content/drive/MyDrive/spam_train_cleaned.csv")
test_df = pd.read_csv("/content/drive/MyDrive/spam_test_cleaned.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)



Train: (31311, 16)
Test: (7828, 16)


In [63]:
same = set(train_df["combined_text"]) & set(test_df["combined_text"])

test_df = test_df[
    ~test_df["combined_text"].isin(same)
].reset_index(drop=True)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

print("Same texts in Train & Test:",
      len(set(train_df["combined_text"]) & set(test_df["combined_text"])))



Train: (27411, 16)
Test: (6647, 16)
Same texts in Train & Test: 0


In [64]:
same = set(train_df["combined_text"]) & set(test_df["combined_text"])

print("Same texts in Train & Test:", len(same))


Same texts in Train & Test: 0


In [44]:
train_texts = set(train_df["combined_text"])

test_df = test_df[
    ~test_df["combined_text"].isin(train_texts)
].reset_index(drop=True)

print("Same texts in Train & Test:",
      len(set(train_df["combined_text"]) & set(test_df["combined_text"])))


Same texts in Train & Test: 0


In [45]:
# check class balance
print("Train class distribution:")
print(train_df["label"].value_counts())

print("\nTest class distribution:")
print(test_df["label"].value_counts())

Train class distribution:
label
1    17461
0    13850
Name: count, dtype: int64

Test class distribution:
label
0    3434
1    3241
Name: count, dtype: int64


In [46]:
print(train_df.columns.tolist())



['label', 'urls', 'hour', 'combined_text', 'capital_letter_count', 'capital_ratio', 'exclamation_count', 'question_count', 'special_char_count', 'day_of_week_Friday', 'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday', 'day_of_week_Tuesday', 'day_of_week_Wednesday']


In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=300)

X_train = tfidf.fit_transform(train_df["combined_text"])
X_test = tfidf.transform(test_df["combined_text"])

y_train = train_df["label"]
y_test = test_df["label"]

print("Train:", X_train.shape)
print("Test:", X_test.shape)


Train: (27411, 300)
Test: (6647, 300)


In [66]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

print("Logistic Regression trained!")


Logistic Regression trained!


In [67]:
from sklearn.metrics import accuracy_score, recall_score, f1_score

train_pred = lr.predict(X_train)
test_pred = lr.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy :", accuracy_score(y_test, test_pred))

print("Train Recall:", recall_score(y_train, train_pred))
print("Test Recall :", recall_score(y_test, test_pred))

print("Train F1:", f1_score(y_train, train_pred))
print("Test F1 :", f1_score(y_test, test_pred))


Train Accuracy: 0.9782933858669877
Test Accuracy : 0.9789378667067851
Train Recall: 0.9823801483004184
Test Recall : 0.983224603914259
Train F1: 0.9782505391673063
Test F1 : 0.9783616692426584


In [76]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, f1_score

lr_c01 = LogisticRegression(C=0.1, max_iter=1000)
lr_c01.fit(X_train, y_train)

train_pred = lr_c01.predict(X_train)
test_pred = lr_c01.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy :", accuracy_score(y_test, test_pred))
print("Train Recall:", recall_score(y_train, train_pred))
print("Test Recall :", recall_score(y_test, test_pred))
print("Train F1:", f1_score(y_train, train_pred))
print("Test F1:", f1_score(y_test, test_pred))

Train Accuracy: 0.9664003502243624
Test Accuracy : 0.9667519181585678
Train Recall: 0.9668159459657881
Test Recall : 0.9655172413793104
Train F1: 0.9662129938735831
Test F1: 0.9656672362901974
